# Practice Lab: Streamlining Preprocessing with Pipelines

In real-world data science, datasets are rarely clean. You will have a mix of text categories, numbers, and missing values. Furthermore, **why** data is missing matters. We cannot simply fill every missing value with the median.

Today, you will learn the industry standard for handling messy data safely and efficiently: **Scikit-Learn's Pipelines** and **ColumnTransformers**. 

### Ames Housing
We are using the famous Ames Housing dataset.The full Ames dataset contains 79 different features detailing almost every physical aspect of a residential home. It is a fantastic playground for machine learning because the data is highly realistic—which means it is messy! 

__Our Focus for Today:__

To keep our focus strictly on mastering Data Preprocessing and Pipelines, we have curated a subset of 12 specific features. This curated list contains a perfect mix of skewness, outliers, and missing values to help you practice handling real-world data without getting overwhelmed by 79 columns!

**The Business Problem:** A real estate firm wants to predict the `SalePrice` of a house based on its characteristics.

**Data Dictionary:**

| Column Name | Definition |
|-------------|------------|
|`GrLivArea`|: Above-ground living area (sq ft). *(Often highly skewed)*|
|`TotalBsmtSF`|: Total square feet of the basement.|
|`OverallQual`|: Rates the overall material and finish (1-10).|
|`YearBuilt`|: Original construction date.|
|`LotFrontage`|: Linear feet of street connected to property. *(Missing values mean the assessor forgot to measure |it).*
|`MasVnrArea`|: Masonry veneer area in sq ft. *(Missing values mean the house does not have a brick/stone facade).*|
|`Neighborhood`|: Physical locations within Ames city limits.|
|`BldgType`|: Type of dwelling (e.g., 1Fam, Townhouse).|
|`CentralAir`|: Central air conditioning (Y/N).|
|`GarageType`|: Location of the garage. *(Missing means "No Garage").*|
|`KitchenQual`|: Kitchen quality (Ex=Excellent, Gd=Good, TA=Typical/Average, Fa=Fair, Po=Poor).|
|`BsmtQual`|: Height/Quality of the basement. *(Missing means "No Basement").*|
|`SalePrice`|: The property's sale price in dollars.|

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import sklearn
sklearn.set_config(transform_output="pandas") # Forces all transformers to output DataFrames!

# Data Loading & Splitting
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, OrdinalEncoder

# Pipelines and Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling & Evaluation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, root_mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data
We will fetch the Ames Housing dataset from OpenML. It has 80 columns, but we will slice it down to our 12 curated features to focus on mastering pipeline architecture.

In [2]:
# Fetch Ames Housing dataset
ames_data = fetch_openml(name="house_prices", as_frame=True, parser='auto')
df = ames_data.frame

# Select our subset of features + target
features_to_keep = [
    'GrLivArea', 'TotalBsmtSF', 'OverallQual', 'YearBuilt', 'LotFrontage', 'MasVnrArea', 
    'Neighborhood', 'BldgType', 'CentralAir', 'GarageType', 'KitchenQual', 'BsmtQual',   
    'SalePrice'
]

apart_df = df[features_to_keep]

apart_df.head()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,LotFrontage,MasVnrArea,Neighborhood,BldgType,CentralAir,GarageType,KitchenQual,BsmtQual,SalePrice
0,1710,856,7,2003,65.0,196.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,208500
1,1262,1262,6,1976,80.0,0.0,Veenker,1Fam,Y,Attchd,TA,Gd,181500
2,1786,920,7,2001,68.0,162.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,223500
3,1717,756,7,1915,60.0,0.0,Crawfor,1Fam,Y,Detchd,Gd,TA,140000
4,2198,1145,8,2000,84.0,350.0,NoRidge,1Fam,Y,Attchd,Gd,Gd,250000


## 2. Exploratory Data Analysis (EDA)

Before we can build our pipelines, we need to understand the shape of our data. Our preprocessing strategy depends entirely on the issues we uncover here.

**Our EDA Checklist:**
1. Check the first 5 rows to get a feel for the data.
2. Use `.info()` and `.isna().sum()` to identify data types and locate missing values.
3. Use `.describe()` for basic summary statistics.
4. Evaluate skewness to choose our mathematical transformations.
5. Evaluate outliers to choose our scaling strategy.
6. Look at the distribution of our text categories.

__1. Display the first 5 rows and look at the data we will be working on__

In [3]:
apart_df.head()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,LotFrontage,MasVnrArea,Neighborhood,BldgType,CentralAir,GarageType,KitchenQual,BsmtQual,SalePrice
0,1710,856,7,2003,65.0,196.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,208500
1,1262,1262,6,1976,80.0,0.0,Veenker,1Fam,Y,Attchd,TA,Gd,181500
2,1786,920,7,2001,68.0,162.0,CollgCr,1Fam,Y,Attchd,Gd,Gd,223500
3,1717,756,7,1915,60.0,0.0,Crawfor,1Fam,Y,Detchd,Gd,TA,140000
4,2198,1145,8,2000,84.0,350.0,NoRidge,1Fam,Y,Attchd,Gd,Gd,250000


__2.1 Get a general information on the data using `.info()`__

In [4]:
apart_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   GrLivArea     1460 non-null   int64  
 1   TotalBsmtSF   1460 non-null   int64  
 2   OverallQual   1460 non-null   int64  
 3   YearBuilt     1460 non-null   int64  
 4   LotFrontage   1201 non-null   float64
 5   MasVnrArea    1452 non-null   float64
 6   Neighborhood  1460 non-null   str    
 7   BldgType      1460 non-null   str    
 8   CentralAir    1460 non-null   str    
 9   GarageType    1379 non-null   str    
 10  KitchenQual   1460 non-null   str    
 11  BsmtQual      1423 non-null   str    
 12  SalePrice     1460 non-null   int64  
dtypes: float64(2), int64(5), str(6)
memory usage: 148.4 KB


__2.2 Check for Missing Values__
>**Note on Missing Values:** In a standard workflow, when you see missing values during EDA, your instinct might be to fix them right away. **Don't!** We are just observing the mess right now. We will handle the actual filling (imputation) dynamically during the Data Preprocessing stage to prevent data leakage.

In [5]:
print(apart_df.isnull().mean()*100)


GrLivArea        0.000000
TotalBsmtSF      0.000000
OverallQual      0.000000
YearBuilt        0.000000
LotFrontage     17.739726
MasVnrArea       0.547945
Neighborhood     0.000000
BldgType         0.000000
CentralAir       0.000000
GarageType       5.547945
KitchenQual      0.000000
BsmtQual         2.534247
SalePrice        0.000000
dtype: float64


__Let's drop the missing values in 'MasVnrArea'__

In [6]:
apart_df = apart_df.dropna(subset='MasVnrArea')
apart_df['MasVnrArea'].isnull().mean()*100

np.float64(0.0)

__3. Look at the Summary statistics using `.describe()`__

In [7]:
apart_df.select_dtypes(include='number').describe()

,GrLivArea,TotalBsmtSF,OverallQual,YearBuilt,LotFrontage,MasVnrArea,SalePrice
count,1452.000000,1452.000000,1452.000000,1452.000000,1195.000000,1452.000000,1452.000000
mean,1514.091598,1055.847107,6.092975,1971.116391,70.030126,103.685262,180615.063361
std,525.627765,438.119089,1.381289,30.193761,24.289276,181.066207,79285.541485
min,334.000000,0.000000,1.000000,1872.000000,21.000000,0.000000,34900.000000
25%,1128.000000,794.750000,5.000000,1954.000000,59.000000,0.000000,129900.000000
50%,1461.500000,990.500000,6.000000,1972.000000,69.000000,0.000000,162700.000000
75%,1776.000000,1297.250000,7.000000,2000.000000,80.000000,166.000000,214000.000000
max,5642.000000,6110.000000,10.000000,2010.000000,313.000000,1600.000000,755000.000000


In [8]:
apart_df.select_dtypes(include='object').describe()

,Neighborhood,BldgType,CentralAir,GarageType,KitchenQual,BsmtQual
count,1452,1452,1452,1371,1452,1415
unique,25,5,2,6,4,4
top,NAmes,1Fam,Y,Attchd,TA,TA
freq,225,1213,1357,863,734,648


>- Look closely at the summary statistics output for TotalBsmtSF and MasVnrArea. Their minimum values are exactly 0. For MasVnrArea, even the 25% and 50% percentiles are 0! Because at least half the houses in Ames simply do not have a masonry veneer, and some don't have basements, we can safely impute (fill) missing values in these columns with 0.
>-  Check out the massive jump between the 75% mark and the max for GrLivArea (1,776 vs. 5,642 sq ft) and SalePrice (214,000 vs. 755,000). That huge gap is a warning sign that we have extreme outliers. If we used MinMaxScaler or StandardScaler, those few huge houses would squish all our normal houses into a tiny, unreadable range.

__4. Evaluate skewness to choose our mathematical transformations__


In [9]:
# We provided this simple function for you:

def skewness_report(data):
    return (
        data.select_dtypes(include="number")
            .skew()
            .sort_values(key=lambda x: x.abs(), ascending=False)
            .to_frame("skewness")
    )


In [10]:
# Run the function
skewness_report(apart_df)

,skewness
MasVnrArea,2.669084
LotFrontage,2.174359
SalePrice,1.884045
TotalBsmtSF,1.533040
GrLivArea,1.374375
YearBuilt,-0.608915
OverallQual,0.214636


__5. Look at the distribution of our text categories__

Finally, let's look at our text features. We want to see how many unique categories exist in each column, which will tell us how wide our dataset will become after One-Hot Encoding.

In [11]:
apart_df.select_dtypes(include='object').nunique()


Neighborhood    25
BldgType         5
CentralAir       2
GarageType       6
KitchenQual      4
BsmtQual         4
dtype: int64

__NOTE:__ 
- *The Neighborhood column contains 25 different categories. If we One-Hot Encode it, it will add 25 new columns to our dataset, which can unnecessarily clutter our simple model. So, for this exercise, let's drop it! However, this does not mean location isn't important!!!*
- *The Basement Quality has around 2.5% missing records and this might be an indicator that the house doesn't have a basement. For this practice, let's say we want to focus on houses with Basement. So, let's drop the rows with missing Basement Quality!*

In [12]:
# Let's drop 'Neighborhood'
apart_df = apart_df.drop(columns='Neighborhood')

In [13]:
# Let's drop the rows with missing Basement Quality
apart_df = apart_df.dropna(subset='BsmtQual')

__Since we dropped the missing Basement Quality, let's evaluate the skewness__

In [14]:
skewness_report(apart_df)

,skewness
MasVnrArea,2.651349
TotalBsmtSF,2.186511
LotFrontage,2.147973
SalePrice,1.893948
GrLivArea,1.389036
YearBuilt,-0.636157
OverallQual,0.254719


## 3. Train-Test Split

Before we do **any** preprocessing (like filling missing values or scaling numbers), we must split our data into training and testing sets. 

**Why? To prevent Data Leakage!**
If we calculate the median `LotFrontage` using the entire dataset, our training process would secretly learn information about the test set. By splitting first, our pipelines will be forced to calculate medians and scaling factors using *only* the training data. The test data remains completely unseen, simulating how a model operates in the real world.

In [15]:
# Separate features (X) and target (y)
X = apart_df.drop(columns='SalePrice')
y = apart_df['SalePrice']

# Perform train-test split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Verify the split

## 4. Part 3: The Hero - ColumnTransformer

To fix the mixed-data problem, we use a **`ColumnTransformer`**. 

If a Pipeline is a single factory assembly line, a ColumnTransformer is the factory manager. It allows you to apply different pipelines to different subsets of your data, and then seamlessly stitches the arrays back together at the end. 

Think of it as a traffic cop routing data based on our business logic:
*   **Branch 1 (Median):** `LotFrontage`, `GrLivArea` $\rightarrow$ Impute Median $\rightarrow$ Scale
*   **Branch 2 (Zero):** `MasVnrArea`, `TotalBsmtSF` $\rightarrow$ Impute Zero $\rightarrow$ Power Transform
*   **Branch 3 (Symmetrical):** `OverallQual`, `YearBuilt` $\rightarrow$ Impute Mean $\rightarrow$ Scale
*   **Branch 4 (OHE Text 1):** `BldgType`, `GarageType` $\rightarrow$ Impute "None" $\rightarrow$ One-Hot Encode
*   **Branch 5 (OHE Text 2):** `CentralAir` $\rightarrow$ Impute "N" $\rightarrow$ One-Hot Encode
*   **Branch 6 (Ordinal Text 1):** `KitchenQual` $\rightarrow$ Impute "Most Frequent" $\rightarrow$ Ordinal Encode $\rightarrow$ Scale
*   **Branch 7 (Ordinal Text 2):** `BsmtQual` $\rightarrow$ Impute "None" $\rightarrow$ Ordinal Encode $\rightarrow$ Scale

__1. Reflect on the imputing choice of each branch__

> **SOLUTION:** 
> 

In [16]:
# First, Define the columns for each branch
num_median_cols = ['LotFrontage', 'GrLivArea'] 
num_zero_cols = ['MasVnrArea', 'TotalBsmtSF']
num_sym_cols = ['OverallQual', 'YearBuilt']
cat1_nominal_cols = ['BldgType', 'GarageType']
cat2_nominal_cols = ['CentralAir']
cat1_ordinal_cols = ['KitchenQual']
cat2_ordinal_cols = ['BsmtQual']

In [17]:
# Then, Build the individual pipelines
## ==> BRANCH 1 (Median): -> Impute Median -> Scale
step1 = ('imputer', SimpleImputer(strategy='median'))
step2 = ('scaler', StandardScaler())

step_list = [step1, step2]
branch1 = Pipeline(steps=step_list)
branch1

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

In [18]:
## ==> BRANCH 2 (Zero): Skewed Numerics -> Impute Zero -> Power Transform
step1 = ('imputer', SimpleImputer(strategy='constant', fill_value=0))
step2 = ('transformer', PowerTransformer(method='yeo-johnson'))

step_list = [step1, step2]

branch2 = Pipeline(steps=step_list)
branch2

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('transformer', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",0
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accoun

In [19]:
## ==> BRANCH 3 (Symmetrical): -> Impute mean -> Standard Scale ONLY
step1 = ('imputer', SimpleImputer(strategy='mean'))
step2 = ('scaler', StandardScaler())

step_list = [step1, step2]
branch3 = Pipeline(steps=step_list)
branch3

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for 

In [20]:
## ==> BRANCH 4 (OHE Text 1): Nominal Categories -> Impute "None" -> One-Hot Encode
step1 = ('imputer', SimpleImputer(strategy='constant', fill_value=None))
step2 = ('ohe', OneHotEncoder(drop='first', sparse_output=False))

step_list = [step1, step2]

branch4 = Pipeline(steps=step_list)
branch4

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('ohe', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for

In [21]:
## ==> BRANCH 5 (OHE Text 2): Nominal Categories -> Impute "N" -> One-Hot Encode
step1 = ('imputer', SimpleImputer(strategy='constant', fill_value='N'))
step2 = ('ohe', OneHotEncoder(drop='first', sparse_output=False))

step_list = [step1, step2]

branch5 = Pipeline(steps=step_list)
branch5

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('ohe', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'N'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for 

In [22]:
## ==> BRANCH 6 (Ordinal Text 1): -> Impute "most-frequent" -> Ordinal Encode -> Scale
step1 = ('imputer', SimpleImputer(strategy='most_frequent'))
step2 = ('ohe', OrdinalEncoder())
step3 = ('scaler', StandardScaler())

step_list = [step1, step2, step3]

branch6 = Pipeline(steps=step_list)
branch6

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('ohe', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto a

In [23]:
## ==> BRANCH 7 (Ordinal Text 2): -> Impute "None" -> Ordinal Encode -> Scale
step1 = ('imputer', SimpleImputer(strategy='constant', fill_value=None))
step2 = ('ohe', OrdinalEncoder())
step3 = ('scaler', StandardScaler())

step_list = [step1, step2, step3]

branch7 = Pipeline(steps=step_list)
branch7

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('ohe', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto accoun

In [24]:
# Finally, Combine all branches into the ColumnTransformer
# all_branches = ColumnTransformer([branch1, branch2, branch3, branch4, branch5, branch6, branch7])
all_branches = ColumnTransformer(transformers=[
    ('Branch1', branch1, num_median_cols),
    ('Branch2', branch2, num_zero_cols),
    ('Branch3', branch3, num_sym_cols),
    ('Branch4', branch4, cat1_nominal_cols),
    ('Branch5', branch5, cat2_nominal_cols),
    ('Branch6', branch6, cat1_ordinal_cols),
    ('Branch7', branch7, cat2_ordinal_cols)
])
all_branches

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Branch1', ...), ('Branch2', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``f

## 5. The Master Pipeline & Evaluation

Now we bundle our `preprocessor` and our `LinearRegression` model into one final **Master Pipeline**. 

Think about all the work we did in Part 1. By using this pipeline, we can apply all of that logic—dropping columns, filling medians, filling zeroes, One-Hot Encoding, Ordinal Encoding, and Power Transforming—with a **single line of code**.

In [27]:
# Helper function to evaluate our models (calculates R-squared and RMSE for train and test sets)

def evaluate_model(model, X_train, y_train, X_test, y_test):
    
    results = {'R2_test':model.score(X_test,y_test),
               'R2_train':model.score(X_train, y_train),
               'RMSE_test':root_mean_squared_error(y_test, model.predict(X_test)),
               'RMSE_train':root_mean_squared_error(y_train, model.predict(X_train))
        
    }

    return results



In [28]:
# 1. Create the final Main Pipeline

main_pipeline = Pipeline([
    ('preprocessor', all_branches),
    ('model', LinearRegression())
])

# 2. Fit the main_pipeline on the RAW training data
# We don't have to use our manually processed data—the pipeline does it all!
main_pipeline.fit(X_train,y_train)

# 3. Evaluate the main_pipeline
evaluate_model(main_pipeline, X_train, y_train, X_test, y_test)

{'R2_test': 0.8160149206400391,
 'R2_train': 0.7946187022338911,
 'RMSE_test': 35457.41203319446,
 'RMSE_train': 35491.05161609278}

## 6. Reflection

**Your Task:**
Double-click this text cell. In 2-3 sentences, explain why passing new, unseen house data through a `Pipeline` is significantly safer for production deployment than the manual approach.

> **SOLUTION:** 
> 

## 7. Communicating Results to Stakeholders

**How to phrase your answer:**
When speaking to stakeholders, avoid throwing raw math at them. Translate the metrics into real-world impact. 

*Example using a Real Estate Model:*
*   ❌ **Bad:** "The model has an RMSE of 25000 and an $R^2$ of 0.85."
*   ✅ **Good:** "Our model is highly accurate, capturing about 85% of the factors that drive house prices. When it makes a prediction, it is typically off by about $25,000 on average."

**Your Task:**
Double-click this text cell. Based on your best model's RMSE and $R^2$ scores, write a 3-4 sentence explanation.
> 